# 02 · ASVspoof 5 — XLSR-MamBo

Primo esperimento allo stato dell'arte sui subset preparati nel notebook 01 (forma d'onda grezza, non le Mel).

**Modello**: XLSR-MamBo, configurazione MamBo-3-Hydra-N3 (Ng et al., *XLSR-MamBo: Scaling the Hybrid
Mamba-Attention Backbone for Audio Deepfake Detection*, arXiv 2601.02944, ACL 2026 Findings;
codice originale https://github.com/saki-ciallo/MamBo-for-ADD, MIT). XLS-R 300M fine-tunato →
5 unità [3×Hydra, MLP, MHA, MLP] → attention pooling → 2 classi. Port in PyTorch puro in `src/thesis/models/`.

**Protocollo**: train sul subset train (5000+5000), scelta dell'epoca **solo** sull'EER del dev
(2500+2500, attacchi A09–A16), test una sola volta sull'eval (5000+5000, attacchi A17–A32 e codec mai visti).
3 seed; metriche EER, minDCF, actDCF, Cllr, EER per attacco e per codec, IC 95% bootstrap dell'EER.

**Differenze dal paper** (dovute alla 3060 da 8 GB / Windows): XLS-R da Hugging Face invece che da torchaudio
(stessi pesi); CNN iniziale di XLS-R congelata e gradient checkpointing; batch 8 × 4 di accumulo (32 effettivo);
focal loss con α = 0.5/0.5 perché i subset sono bilanciati; nessuna RawBoost (esperimento separato);
training sui subset di ASVspoof 5, non su ASVspoof 2019 LA.

Il training gira **fuori da Jupyter** (`python -m thesis.train`), così non si ferma se VS Code si scollega.
Questo notebook lo lancia, lo monitora e analizza i risultati salvati su file.

## 0 · Setup

In [ ]:
import json
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # il notebook gira da notebooks/
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis import config, report
from thesis.train import TrainConfig

print(config.describe())

## 1 · Configurazione

In [ ]:
# Esperimenti XLSR-MamBo seguiti e confrontati in questo notebook: nome → seed
RUNS = {
    "mambo_asvspoof5": [1, 2, 3],           # baseline
    "mambo_rawboost_asvspoof5": [1, 2, 3],  # + RawBoost (algoritmo 5) sul train
    "nes2net_x_rawboost_asvspoof5": [1],    # confronto: WavLM-Large + Nes2Net-X con RawBoost
}
# Config da lanciare nella sezione 2 (una di quelle sopra)
CONFIG = REPO / "configs" / "mambo_rawboost_asvspoof5.json"

cfg = TrainConfig.from_json(CONFIG)
SEEDS = RUNS[cfg.name]
for name in RUNS:
    c = json.loads((REPO / "configs" / f"{name}.json").read_text())
    print(f"{name}: rawboost={c.get('rawboost')}, patience={c['patience']}, seed={RUNS[name]}")

## 2 · Lancio del training in background

Avvia un processo separato e torna subito: il training continua anche chiudendo VS Code.
I seed già completati vengono saltati, quello interrotto riprende dall'ultima epoca salvata.
**Una sola volta**: due training insieme non stanno in 8 GB (la sezione 3 dice se ce n'è già uno attivo).

In [ ]:
LAUNCH = False  # metti True per avviare

if LAUNCH:
    assert not report.processo_attivo(CONFIG.name), "c'è già un training attivo con questa config"
    pid = report.lancia([CONFIG.relative_to(REPO)], SEEDS, f"{cfg.name}/train.log")
    print("Avviato, PID", pid)

## 3 · Monitoraggio

Legge `train.log` e i `log.csv` senza toccare il training: seed ed epoca correnti, batch fatti/totali,
velocità, tempo alla fine dell'epoca, EER dev delle epoche finite. Un riquadro **ATTENZIONE** segnala
processo non attivo, log fermo da più di 10 minuti, loss NaN/inf o errori nel log.
Con `AGGIORNA_MIN > 0` si aggiorna da sola ogni minuto (si ferma con ■; il training non ne risente).

In [ ]:
AGGIORNA_MIN = 0  # es. 60 = aggiorna ogni minuto per un'ora

def stato_mambo():
    for name, seeds in RUNS.items():
        report.stato(name, f"{name}.json", seeds, report.run_dir(name) / "train.log")
        print()

report.ogni_minuto(stato_mambo, AGGIORNA_MIN)

## 4 · Curve di training (una riga per esperimento)

In [ ]:
fig, axes = plt.subplots(len(RUNS), 2, figsize=(12, 3.2 * len(RUNS)), squeeze=False)
for row, (name, seeds) in zip(axes, RUNS.items()):
    for seed in seeds:
        log_path = report.run_dir(name) / f"seed{seed}" / "log.csv"
        if not log_path.exists():
            continue
        log = pd.read_csv(log_path)
        row[0].plot(log.epoca, log.train_loss, label=f"train s{seed}")
        row[0].plot(log.epoca, log.dev_loss, "--", label=f"dev s{seed}")
        row[1].plot(log.epoca, 100 * log.dev_eer, marker="o", label=f"seed {seed}")
    row[0].set(title=f"{name} — loss (train: focal, dev: CE)", xlabel="epoca", yscale="log")
    row[1].set(title=f"{name} — EER dev (%)", xlabel="epoca")
    for ax in row:
        if ax.lines:
            ax.legend(fontsize=7)
plt.tight_layout()

## 5 · Risultati

Tabelle generate dai `metrics.json` (solo i seed completati). actDCF e Cllr misurano la calibrazione
dei punteggi (logit bonafide − logit spoof, senza calibrazione a posteriori).

In [ ]:
per_seed = report.risultati(list(RUNS), sorted({s for v in RUNS.values() for s in v}))
assert len(per_seed), "Nessun seed completato: guarda la sezione 3"
done = per_seed.groupby("run").seed.unique().to_dict()  # seed completati per esperimento
display(per_seed.round(4))
display(report.riepilogo(per_seed))

### Confronto a parità di seed

Il confronto più pulito tra esperimenti è sugli stessi seed (es. RawBoost seed 1 contro baseline seed 1):
la variabilità tra seed della baseline è grande quanto le differenze che cerchiamo.

In [ ]:
comuni = sorted(int(s) for s in set.intersection(*(set(v) for v in done.values())))
print("Seed in comune:", comuni)
if comuni:
    pari = per_seed[per_seed.seed.isin(comuni)]
    display(pari.pivot_table(index=["split", "seed"], columns="run", values="EER %").round(2))

### EER per attacco e per codec (eval, media sui seed completati)

In [ ]:
attacks = pd.DataFrame({n: report.eer_per_gruppo(n, list(s), "eer_per_attacco")["media"] for n, s in done.items()})
codecs = pd.DataFrame({n: report.eer_per_gruppo(n, list(s), "eer_per_codec")["media"] for n, s in done.items()})
display(attacks)
display(codecs)

fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={"width_ratios": [2, 1]})
attacks.plot.bar(ax=axes[0], title="EER per attacco — eval (%)")
codecs.plot.bar(ax=axes[1], title="EER per codec — eval ('-' = nessuno) (%)", legend=False)
plt.tight_layout()

### Distribuzione dei punteggi (eval, primo seed completato di ogni esperimento)

In [ ]:
fig, axes = plt.subplots(1, len(done), figsize=(7 * len(done), 3.5), squeeze=False)
for ax, (name, seeds) in zip(axes[0], done.items()):
    scores = pd.read_csv(report.run_dir(name) / f"seed{seeds[0]}" / "scores_eval.csv")
    for label, g in scores.groupby("label"):
        ax.hist(g["score"], bins=80, alpha=0.6, density=True, label=label)
    ax.axvline(-np.log(0.95 / 0.5), color="k", ls="--", lw=1, label="soglia actDCF")
    ax.set(title=f"{name} — seed {seeds[0]}", xlabel="logit bonafide − logit spoof"); ax.legend()
plt.tight_layout()

## 6 · Calibrazione dei punteggi

Regressione logistica lineare (llr = a·punteggio + b, classi bilanciate) stimata **sul dev** e applicata all'eval.
EER e minDCF non cambiano (controllato con un assert); cambiano actDCF e Cllr, che misurano se i punteggi
sono utilizzabili come log-likelihood ratio. I punteggi calibrati vanno in `scores_<split>_calibrati.csv`,
i parametri in `calibrazione.json`; gli originali non vengono toccati.

In [ ]:
cal = pd.concat([report.calibra(n, [int(s) for s in seeds]).assign(run=n) for n, seeds in done.items()])
display(cal.round(4))
display(cal[cal.split == "eval"].groupby("run")[["actDCF prima", "actDCF dopo", "Cllr prima", "Cllr dopo"]]
        .agg(["mean", "std"]).round(4))